In [10]:
import pandas as pd
import numpy as np

TARGETS = ["theta"]

In [11]:
results_1l = pd.read_excel("resultados-1l.xlsx")
results_2l = pd.read_excel("resultados-2l.xlsx")
results_3l = pd.read_excel("resultados-3l.xlsx")

results = pd.concat(
    [results_1l, results_2l, results_3l],
    ignore_index=True
)

In [14]:
# 🔹 categorização dos sets (baseada nos comentários originais)

SETS_CATEGORY = {
    "ZZx1":  "Train",
    "ZZx2":     "Val",
    "ZZxReto":  "Test",
    "LSG-1":    "Test",
    
    "ZZy1":     "Test",
    "ZZy2":     "Test",
    "LSG-2":    "Test",
    "ZZx1-inv": "Test",
    "ZZx2-inv": "Test",
    "semiCirc": "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 5  # top modelos

w_val = 0.0
w_train = 1
w_test = 0.0

for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"]
        # - 0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 5 MODELOS - theta


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
7467,model_arch94-41-23_r0.01_Ld0.7_Lp0.3_seed7869,"[94, 41, 23]",0.982818,0.614511,-12.308754,0.982818
4742,model_arch49-24_r0.9_Ld0.7_Lp0.3_seed5928,"[49, 24]",0.981768,-0.199227,-11.935387,0.981768
5370,model_arch54-27_r0.9_Ld0.7_Lp0.3_seed1716,"[54, 27]",0.980439,0.771050,-10.350805,0.980439
1077,model_arch36_r0.9_Ld0.7_Lp0.3_seed7320,[36],0.980073,0.646480,-13.204912,0.980073
5579,model_arch54-34_r0.01_Ld0.7_Lp0.3_seed5071,"[54, 34]",0.977487,0.544746,-13.576996,0.977487



📊 MÉTRICAS COMPLETAS - TOP 5 (theta)


,model,Neurons,R2_ZZx1_theta,R2_ZZx2_theta,R2_ZZxReto_theta,R2_LSG_1_theta,R2_ZZy1_theta,R2_ZZy2_theta,R2_LSG_2_theta,R2_ZZx1_inv_theta,R2_semiCirc_theta,R2_train_mean,R2_val_mean,R2_test_mean,Score
7467,model_arch94-41-23_r0.01_Ld0.7_Lp0.3_seed7869,"[94, 41, 23]",0.982818,0.614511,0.922009,-0.647709,-20.976380,-4.818226,-2.130433,-0.738628,-57.771913,0.982818,0.614511,-12.308754,0.982818
4742,model_arch49-24_r0.9_Ld0.7_Lp0.3_seed5928,"[49, 24]",0.981768,-0.199227,0.614009,-0.485807,-24.403739,-2.578540,-1.227958,0.313878,-55.779552,0.981768,-0.199227,-11.935387,0.981768
5370,model_arch54-27_r0.9_Ld0.7_Lp0.3_seed1716,"[54, 27]",0.980439,0.771050,0.743641,0.387470,-18.090372,-5.314752,-1.858511,-0.886205,-47.436909,0.980439,0.771050,-10.350805,0.980439
1077,model_arch36_r0.9_Ld0.7_Lp0.3_seed7320,[36],0.980073,0.646480,0.911628,-0.334702,-23.986463,-9.221766,-2.438578,-1.319800,-56.044704,0.980073,0.646480,-13.204912,0.980073
5579,model_arch54-34_r0.01_Ld0.7_Lp0.3_seed5071,"[54, 34]",0.977487,0.544746,0.876028,-0.397547,-22.606628,-10.406167,-3.963441,-0.338020,-58.203198,0.977487,0.544746,-13.576996,0.977487


In [17]:
final_table.to_excel("BestModels-otm.xlsx")